**News Sentiment as a Trading Signal: Measuring Predictive Decay Across Holding Horizons**

Syed Sirajuddin · Master of Science in Applied Artificial Intelligence · Shiley Marcos School of Engineering, University of San Diego · AAI-590 Capstone

# Notebook 1 of 5 — Introduction, Data Acquisition, and Cleaning

This notebook series is organized to mirror both the required elements of the capstone code base and the sections of the final report. The mapping is as follows, so that a reviewer can locate each required element directly:

| Notebook | Code-base requirement | Report section(s) |
| --- | --- | --- |
| 01 — this notebook | Data Cleaning | Introduction; Data Summary |
| 02 — Exploratory Data Analysis | Exploratory Data Analysis | Data Summary |
| 03 — Sentiment Model Fine-Tuning | Model Training | Literature Review; Methodology |
| 04 — Signals & Multi-Horizon Backtesting | Model/Pipeline Design and Building | Methodology |
| 05 — Optimization & Analysis | Model Optimization; Analysis and Discussion | Methodology; Results; Conclusion |

Reusable logic lives in the `src/` package and is imported here rather than duplicated, so that the notebooks remain readable while the repository retains a single, tested implementation of each component. Results-oriented commentary is deferred: cells marked *"Interpretation — to be completed"* will be filled in once the models have been trained and tested on live data.


## 1. Introduction

Modern equity markets generate a continuous, high-volume stream of price-relevant news — earnings releases, analyst rating changes, product announcements, and regulatory actions — far more than any individual investor can read and act on in a timely way. Under the efficient market hypothesis, public information should be incorporated into prices almost immediately (Fama, 1970); empirically, however, a substantial literature documents that the textual tone of news predicts returns with a measurable lag. Tetlock (2007) showed that media pessimism predicts downward pressure on prices followed by reversion, and Heston and Sinha (2017) found that the horizon of predictability depends strongly on how sentiment is aggregated, with daily news sentiment predicting returns over one to two days but weekly aggregation extending predictability to a quarter.

This project asks a deliberately comparative question: **if daily financial-news sentiment carries tradable information, over what holding horizon does that information remain exploitable?** Rather than committing to a single trading style, we treat the holding horizon as the primary experimental variable and evaluate the *same* sentiment-derived signals at holds ranging from one day to six months, against buy-and-hold and no-skill baselines. Our hypothesis, motivated by the reversal and decay patterns in the literature cited above, is that predictive value is strongest at short (swing) horizons and decays as the holding period lengthens. A finding that the signal disappears beyond a certain horizon is as informative as a finding that it persists.

The intended end user is a retail investor who actively manages a portfolio and needs a research and decision-support screener — not a black-box autopilot, and not a high-frequency system. In a deployed setting, the pipeline built here would consume streaming news and live market data; for research and backtesting, we reconstruct both streams historically on a strict point-in-time basis.

Two data sources feed the project: (1) daily open-high-low-close-volume (OHLCV) bars for a universe of twenty liquid, large-capitalization U.S. equities, obtained through the open-source `yfinance` library (Aroussi, 2023); and (2) historical financial-news headlines linked to those tickers, obtained from the Finnhub company-news API with the open GDELT project as a longer-history fallback. A third, auxiliary dataset — the Financial PhraseBank of roughly 4,800 expert-annotated sentences (Malo, Sinha, Korhonen, Wallenius, & Takala, 2014) — is used in Notebook 03 to fine-tune and validate the sentiment model.


In [1]:
#!python -m pip install -r requirements.txt

In [2]:
#!python -m pip uninstall -y torch
#!python -m pip install torch --index-url https://download.pytorch.org/whl/cu124 # Only for Windows with CUDA 12.4
#!python -m pip install python-dotenv -q

In [3]:
# Environment setup: resolve the repository root so `src` imports work
# whether this notebook is run from notebooks/ or the project root.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
RANDOM_SEED = 42

Project root: d:\assignments\Assignments\AAI590\FinalProject


In [4]:
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

## 2. Price Data: Universe, Acquisition, and Cleaning

The universe consists of twenty large-cap U.S. names spanning technology, financials, health care, energy, consumer, and industrial sectors. Two considerations drive this choice. First, news-to-ticker linkage — the noisiest step in any news-based pipeline — is far more reliable for heavily covered large caps than for small caps, where sparse and ambiguous coverage would contaminate the signal. Second, high liquidity keeps the simple transaction-cost model used in backtesting (a fixed per-side charge) defensible; for illiquid names, market impact would dominate and require a more elaborate cost model.

The date range (January 2020 through December 2025) covers several distinct market regimes — the 2020 pandemic crash and recovery, the 2022 drawdown, and subsequent recoveries — which matters because a sentiment signal that only works in one regime is of limited practical value.


In [5]:
from src.config import UNIVERSE
from src.data.prices import download_prices, clean_prices

print(f"Universe ({len(UNIVERSE.tickers)} tickers): {', '.join(UNIVERSE.tickers)}")
print(f"Date range: {UNIVERSE.start_date} to {UNIVERSE.end_date}")

# Download raw OHLCV bars (writes data/raw/prices_raw.parquet)
prices_raw = download_prices()
prices_raw.head()

Universe (20 tickers): AAPL, MSFT, GOOGL, AMZN, META, NVDA, TSLA, JPM, JNJ, XOM, WMT, PG, V, UNH, HD, DIS, NFLX, AMD, BA, PFE
Date range: 2020-01-01 to 2025-12-31


2026-07-13 21:21:25,217 Downloaded 30140 rows for 20 tickers


Price,date,open,high,low,close,adj_close,volume,ticker
0,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.333878,135480400,AAPL
1,2020-01-03,74.287498,75.144997,74.125000,74.357498,71.630646,146322800,AAPL
2,2020-01-06,73.447502,74.989998,73.187500,74.949997,72.201408,118387200,AAPL
3,2020-01-07,74.959999,75.224998,74.370003,74.597504,71.861847,108872000,AAPL
4,2020-01-08,74.290001,76.110001,74.290001,75.797501,73.017838,132079200,AAPL


### 2.1 Cleaning rules and their rationale

Four cleaning rules are applied, each targeting a data-quality issue with a known source. In a fully productionized pipeline, each of these checks would run as an automated data-validation stage (with alerting) rather than as a one-time cleaning step:

1. **Missing OHLC values** are dropped. These are rare and typically correspond to trading halts or vendor gaps; imputation would fabricate prices on days when no trade was possible.
2. **Single-day volume gaps** are forward-filled; longer gaps are dropped, since extended missing volume usually indicates a data-vendor problem rather than a market event.
3. **Insufficient history** — tickers with fewer than one year of observations are excluded, because the longest holding horizon studied (126 trading days) requires substantial history for even a handful of non-overlapping trades.
4. **Extreme single-day returns** (beyond ±50%) that are *not* corroborated by a volume spike are winsorized. Genuine 50% moves are accompanied by heavy trading; a "quiet" extreme move is almost always a bad print or an unadjusted split artifact from the vendor.

All return computations downstream use the **split- and dividend-adjusted close**, so that corporate actions do not masquerade as price moves.


In [6]:
prices = clean_prices(prices_raw)
print(f"Clean panel: {len(prices):,} rows, "
      f"{prices['ticker'].nunique()} tickers, "
      f"{prices['date'].min().date()} to {prices['date'].max().date()}")
prices.groupby("ticker")["date"].agg(["min", "max", "count"])

2026-07-13 21:21:25,234 Dropped 0 rows with missing OHLC


Clean panel: 30,140 rows, 20 tickers, 2020-01-02 to 2025-12-30


,min,max,count
ticker,,,
AAPL,2020-01-02,2025-12-30,1507
AMD,2020-01-02,2025-12-30,1507
AMZN,2020-01-02,2025-12-30,1507
BA,2020-01-02,2025-12-30,1507
DIS,2020-01-02,2025-12-30,1507
GOOGL,2020-01-02,2025-12-30,1507
HD,2020-01-02,2025-12-30,1507
JNJ,2020-01-02,2025-12-30,1507
JPM,2020-01-02,2025-12-30,1507


## 3. News Data: Acquisition, Deduplication, and Point-in-Time Alignment

Headlines are pulled per ticker from the Finnhub company-news endpoint (chunked by month to respect free-tier rate limits), with the GDELT DOC API available as an alias-matched fallback when a longer history is required. Every article is normalized to a common schema: publication timestamp (UTC), ticker, headline, summary, source, and URL.

Two cleaning steps follow. **Deduplication** removes syndicated wire copies — the same headline republished by many outlets — which would otherwise let a single news event masquerade as many independent signals and overweight it in the daily aggregate. **Length filtering** drops fragments under roughly ten characters, which are almost always feed artifacts rather than headlines.

### 3.1 The point-in-time guarantee

The single most important design decision in this notebook is the mapping of every article to an **effective date**: the first trading session on which a trader could have acted on it at the close. An article published during Tuesday's session is effective Tuesday; an article published after Tuesday's close, on a weekend, or on a holiday is effective the *next* trading session. This mapping is applied once, at ingestion, so every downstream stage — aggregation, signal generation, backtesting — inherits the guarantee automatically instead of re-implementing it (and risking inconsistency).

Look-ahead bias of exactly this kind — evaluating a strategy using information that was not yet available at decision time — is among the most common ways published backtests overstate performance (Bailey, Borwein, López de Prado, & Zhu, 2014; López de Prado, 2018). Because our longest horizons stretch to six months, even small alignment errors would compound across the holding period, which is why the discipline is enforced at the earliest possible stage.


In [7]:
import os
from src.data.news import fetch_finnhub, clean_news
from src.config import RAW_DIR

# Cache the raw news data so we don't have to hit the API every time we run this notebook.
cache = RAW_DIR / "news_finnhub.parquet"

# Requires FINNHUB_API_KEY in the environment.
assert os.environ.get("FINNHUB_API_KEY"), "Set FINNHUB_API_KEY before running."

print("Fetching news from Finnhub... using FINNHUB_API_KEY from environment.")

if cache.exists():
    news_raw = pd.read_parquet(cache)
    print(f"Loaded {len(news_raw):,} cached articles")
else:
    news_raw = fetch_finnhub()

print(f"Raw articles: {len(news_raw):,}")

trading_days = pd.DatetimeIndex(prices["date"].drop_duplicates().sort_values())
news = clean_news(news_raw, trading_days)
print(f"After dedup/length filter: {len(news):,}")
news[["published_at", "effective_date", "ticker", "headline"]].head(8)

Fetching news from Finnhub... using FINNHUB_API_KEY from environment.
Loaded 26,955 cached articles
Raw articles: 26,955
After dedup/length filter: 26,016


,published_at,effective_date,ticker,headline
0,2025-07-21 17:16:18,2025-07-21,AMD,"Sanmina (SANM) Stock Trades Up, Here Is Why"
1,2025-07-21 16:53:00,2025-07-21,AMD,Taiwan Semi Joins the $1 Trillion Club. What’s...
2,2025-07-21 15:26:58,2025-07-21,AMD,S&P 500 and Nasdaq 100 Jump to Record Highs as...
3,2025-07-21 15:19:00,2025-07-21,AMD,Intel Gears Up to Report Q2 Earnings: Should Y...
4,2025-07-21 13:59:00,2025-07-21,AMD,Nvidia Stock Gains. Why Big Tech Earnings Can ...
5,2025-07-21 13:35:16,2025-07-21,AMD,10 Information Technology Stocks With Whale Al...
6,2025-07-21 12:38:08,2025-07-21,AMD,"Advanced Micro Devices, Inc. (AMD): I’m Not Be..."
7,2025-07-21 12:15:00,2025-07-21,AMD,Wall Street Lunch: Charles Schwab Expands Over...


### 3.2 Auxiliary dataset: Financial PhraseBank

The Financial PhraseBank (Malo et al., 2014) contains approximately 4,800 sentences drawn from financial news, each labeled *positive*, *neutral*, or *negative* by annotators with finance backgrounds, at four inter-annotator agreement levels. We use the 75%-agreement subset, trading a modest reduction in size for substantially cleaner labels, and split it 80/10/10 (train/validation/test) with stratification by class. This corpus serves two roles in Notebook 03: fine-tuning the transformer sentiment model and providing a held-out benchmark of its classification quality before it is trusted to score live headlines.


In [8]:
from src.data.phrasebank import load_phrasebank

splits = load_phrasebank(seed=RANDOM_SEED)
for name, df in splits.items():
    print(f"{name:>5}: {len(df):>5} sentences | "
          f"class balance: {df['label'].value_counts(normalize=True).round(2).to_dict()}")

c:\Users\SyedM\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-13 21:21:26,454 HTTP Request: HEAD https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip "HTTP/1.1 302 Found"


train:  2762 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}
  val:   345 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}
 test:   346 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}


## 4. Summary and Hand-off

At the end of this notebook, three clean artifacts exist under `data/`: the cleaned price panel (`prices_clean.parquet`), the deduplicated and point-in-time-aligned news table, and the stratified PhraseBank splits. Notebook 02 explores both market and news streams before any modeling; Notebook 03 trains the sentiment model.

**Data quality summary — to be completed after the acquisition run:** counts of rows dropped by each cleaning rule, per-ticker article coverage, and any vendor-specific anomalies encountered will be recorded here once live acquisition has executed, and will feed the Data Summary section of the final report.


---
### Acknowledgment of AI Tool Use

Portions of the code scaffolding and prose in this notebook were drafted with the assistance of Anthropic's Claude (Anthropic, 2026) and subsequently reviewed, tested, and revised by the author, who takes full responsibility for the final content, design decisions, and results. This acknowledgment is provided in accordance with University of San Diego academic integrity guidelines on the use of generative AI tools.

Anthropic. (2026). *Claude* [Large language model]. https://claude.ai

### References

Aroussi, R. (2023). *yfinance* [Computer software]. https://github.com/ranaroussi/yfinance

Bailey, D. H., Borwein, J. M., López de Prado, M., & Zhu, Q. J. (2014). Pseudo-mathematics and financial charlatanism: The effects of backtest overfitting on out-of-sample performance. *Notices of the American Mathematical Society, 61*(5), 458–471.

Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *The Journal of Finance, 25*(2), 383–417.

Heston, S. L., & Sinha, N. R. (2017). News vs. sentiment: Predicting stock returns from news stories. *Financial Analysts Journal, 73*(3), 67–83.

López de Prado, M. (2018). *Advances in financial machine learning.* Wiley.

Malo, P., Sinha, A., Korhonen, P., Wallenius, J., & Takala, P. (2014). Good debt or bad debt: Detecting semantic orientations in economic texts. *Journal of the Association for Information Science and Technology, 65*(4), 782–796.

Tetlock, P. C. (2007). Giving content to investor sentiment: The role of media in the stock market. *The Journal of Finance, 62*(3), 1139–1168.
